# Régression

Le descriptif a montré que la pauvreté seule est presque sans lien avec l'obésité, et que la région pèse beaucoup. Ce notebook teste ce qu'il reste des liens une fois la région prise en compte.

Les résultats décrivent des associations entre départements, pas des causes.

## 1. Chargement

In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("../data/processed/departements.csv", dtype={"dep": str, "region": str})
df["log_densite"] = np.log(df["densite"])
df.shape

(96, 18)

On prend le logarithme de la densité : elle va de 15 à près de 20 000 hab/km², et sans le logarithme Paris dominerait le modèle.

## 2. Pauvreté seule, puis pauvreté avec la région

In [9]:
modele1 = smf.ols("obesite ~ taux_pauvrete", data=df).fit(cov_type="HC3")

resultats1 = pd.DataFrame({
    "coefficient": modele1.params,
    "ecart_type": modele1.bse,
    "p_valeur": modele1.pvalues,
})
print("R2 :", round(modele1.rsquared, 3))
resultats1.round(3)

R2 : 0.006


,coefficient,ecart_type,p_valeur
Intercept,14.610,1.510,0.000
taux_pauvrete,0.083,0.102,0.418


**Interprétation :** sans la région, le coefficient de la pauvreté est de 0,08 point d'obésité par point de pauvreté, mais il n'est pas significatif (p = 0,42) et le R² est de 0,6 % : la pauvreté seule n'explique presque rien. C'est cohérent avec la corrélation de 0,10 du descriptif.

On ajoute maintenant la région comme effet fixe (`C(region)`), c'est-à-dire qu'on compare les départements au sein de leur région. Les écarts-types sont robustes (`HC3`), c'est-à-dire moins sensibles aux départements atypiques.

In [10]:
modele2 = smf.ols("obesite ~ taux_pauvrete + C(region)", data=df).fit(cov_type="HC3")

resultats2 = pd.DataFrame({
    "coefficient": modele2.params,
    "ecart_type": modele2.bse,
    "p_valeur": modele2.pvalues,
})
print("R2 :", round(modele2.rsquared, 3))
resultats2.loc[["taux_pauvrete"]].round(3)

R2 : 0.589


,coefficient,ecart_type,p_valeur
taux_pauvrete,0.316,0.107,0.003


In [11]:
modele_region = smf.ols("obesite ~ C(region)", data=df).fit()
print("R2 de la région seule :", round(modele_region.rsquared, 3))
print("R2 ajusté du modèle avec pauvreté et région :", round(modele2.rsquared_adj, 3))
print("R2 ajusté de la région seule :", round(modele_region.rsquared_adj, 3))

R2 de la région seule : 0.535
R2 ajusté du modèle avec pauvreté et région : 0.524
R2 ajusté de la région seule : 0.468


**Interprétation :** avec la région, le coefficient de la pauvreté passe à 0,32 point d'obésité par point de pauvreté. Il est significatif (p = 0,003, intervalle de confiance à 95 % de 0,11 à 0,53). À région égale, un département qui a 1 point de pauvreté de plus a en moyenne 0,3 point d'obésité de plus. Le lien n'apparaissait pas dans le descriptif parce que la région le masquait.

Le R² monte à 0,59, mais la région seule en explique déjà 0,54 : la région fait presque tout le travail, et la pauvreté ajoute environ 5 points de R². Son effet est réel mais modeste.

## 3. Modèle complet

On ajoute les autres variables du sujet : l'accès aux médecins (`apl`), les fast-foods et les salles de sport pour 10 000 habitants, la densité (en logarithme) et la part des 60 ans et plus. La région reste en effet fixe.

On garde les fast-foods et on laisse de côté la restauration : les fast-foods sont déjà comptés dans la restauration, donc les deux variables se ressemblent trop (corrélation de 0,67).

In [12]:
formule = (
    "obesite ~ taux_pauvrete + apl + fastfoods_10k + salles_sport_10k"
    " + log_densite + part_60_plus + C(region)"
)
modele3 = smf.ols(formule, data=df).fit(cov_type="HC3")

resultats3 = pd.DataFrame({
    "coefficient": modele3.params,
    "ecart_type": modele3.bse,
    "p_valeur": modele3.pvalues,
    "ic_bas": modele3.conf_int()[0],
    "ic_haut": modele3.conf_int()[1],
})
print("R2 :", round(modele3.rsquared, 3), "| R2 ajusté :", round(modele3.rsquared_adj, 3))
resultats3[~resultats3.index.str.startswith("C(")].round(3)

R2 : 0.735 | R2 ajusté : 0.673


,coefficient,ecart_type,p_valeur,ic_bas,ic_haut
Intercept,13.024,4.818,0.007,3.580,22.468
taux_pauvrete,0.302,0.093,0.001,0.119,0.484
apl,-0.343,0.630,0.586,-1.578,0.891
fastfoods_10k,-0.425,0.166,0.011,-0.751,-0.099
salles_sport_10k,-0.652,0.461,0.157,-1.555,0.251
log_densite,-0.333,0.432,0.441,-1.179,0.514
part_60_plus,0.124,0.092,0.180,-0.057,0.304


**Interprétation :**

- **Pauvreté :** le coefficient reste à 0,30 (p = 0,001), presque comme dans le modèle avec la région seule (0,32). Le lien tient quand on ajoute les autres variables.
- **Fast-foods :** le coefficient est négatif (-0,43, p = 0,011) : à région égale, un fast-food de plus pour 10 000 habitants correspond à 0,4 point d'obésité en moins. C'est le contraire de l'hypothèse de départ. Comme dans le descriptif, ça reflète sans doute le profil des territoires (villes, tourisme) et pas un effet protecteur. On vérifiera si ce résultat dépend de Paris.
- **APL, salles de sport, densité, part des 60 ans et plus :** aucun n'est significatif (p supérieur à 0,15), et leurs intervalles de confiance contiennent 0. On ne peut pas dire que ces variables sont liées à l'obésité une fois la région prise en compte.
- Le R² est de 0,74 (R² ajusté 0,67).

## 4. Comparaison des trois modèles

In [13]:
comparaison = pd.DataFrame(
    {
        "coefficient": [m.params["taux_pauvrete"] for m in (modele1, modele2, modele3)],
        "p_valeur": [m.pvalues["taux_pauvrete"] for m in (modele1, modele2, modele3)],
        "R2_ajuste": [m.rsquared_adj for m in (modele1, modele2, modele3)],
    },
    index=["Pauvreté seule", "Avec la région", "Modèle complet"],
)
comparaison.round(3)

,coefficient,p_valeur,R2_ajuste
Pauvreté seule,0.083,0.418,-0.005
Avec la région,0.316,0.003,0.524
Modèle complet,0.302,0.001,0.673


**Interprétation :** l'effet de la pauvreté n'apparaît que quand on tient compte de la région, et il reste stable (autour de 0,3) quand on ajoute les autres variables. Le R² ajusté passe de 0 à 0,52 puis à 0,67.